In [ ]:
!pip uninstall numpy scipy -y
%pip install numpy==1.26.4 scipy==1.10.1

^C


In [ ]:
%pip install langchain langchain-community langchain-text-splitters pypdf


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


^C
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from datetime import datetime
import os

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [1]:
# Exercise 1 — Load PDFs

print("Loading PDFs...\n")

pdf1_path = "pdfs/Complaint Detection.pdf"
pdf2_path = "pdfs/Complaint-and-Severity-Identification-from-Online-Financial-Content.pdf"

loader1 = PyPDFLoader(pdf1_path)
loader2 = PyPDFLoader(pdf2_path)

docs1 = loader1.load()
docs2 = loader2.load()

all_docs = docs1 + docs2

print(f"PDF 1 Pages: {len(docs1)}")
print(f"PDF 2 Pages: {len(docs2)}")
print(f"Total Documents: {len(all_docs)}")

Loading PDFs...



NameError: name 'PyPDFLoader' is not defined

In [ ]:
# Exercise 2 — Split into Chunks

print("\nSplitting into chunks...\n")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(all_docs)

print(f"Total Chunks Created: {len(chunks)}")

In [ ]:
# Exercise 3 — Attach Metadata

print("\nAdding metadata...\n")

for chunk in chunks:
    source_path = chunk.metadata.get("source", "")
    filename = os.path.basename(source_path)

    chunk.metadata["filename"] = filename
    chunk.metadata["page_number"] = chunk.metadata.get("page", 0)
    chunk.metadata["upload_date"] = datetime.now().strftime("%Y-%m-%d")

    if "Complaint Detection" in filename:
        chunk.metadata["source_type"] = "conference_paper"
    else:
        chunk.metadata["source_type"] = "research_paper"

print("Metadata added successfully!\n")
print("Sample Metadata:\n", chunks[0].metadata)

In [ ]:
# Exercise 4 — Filter Function

def filter_chunks(chunks, **filters):
    result = []

    for chunk in chunks:
        match = True
        for key, value in filters.items():
            if chunk.metadata.get(key) != value:
                match = False
                break

        if match:
            result.append(chunk)

    return result

print("Filter function created!")

In [ ]:
# Exercise 5 — Test Filters

print("\nTesting filters...\n")

# Filter by filename
file_chunks = filter_chunks(chunks, filename="Complaint Detection.pdf")
print(f"Chunks from Complaint Detection.pdf: {len(file_chunks)}")

# Filter by page number
page_chunks = filter_chunks(chunks, page_number=0)
print(f"Chunks from Page 0: {len(page_chunks)}")

# Filter by source type
type_chunks = filter_chunks(chunks, source_type="research_paper")
print(f"Chunks from Research Papers: {len(type_chunks)}")

# Show sample content
if file_chunks:
    print("\nSample Chunk:\n")
    print(file_chunks[0].page_content[:300])